# B3-J1 Atelier 2 -- Mini-pipeline data (Pandas -> PySpark)

**Cours** : Big Data B3 -- Jour 1 (20 mai 2026)
**Duree** : 1h15
**Travail** : en binomes

---

## Objectifs

Construire un pipeline de bout en bout, **squelette de votre projet fil rouge** des 4 jours :

```
2 sources (CSV + JSON) -> Chargement -> Exploration -> Nettoyage
-> Jointure -> Analyse -> Visualisation -> [BONUS B3 : PySpark + .explain()]
```

C'est exactement ce que fait un Data Engineer au quotidien.

**Cellules avec `# A completer`** : a vous de jouer.

**Datasets** : Spotify top tracks (CSV) + profils utilisateurs synthetiques (JSON embedded).

## Setup : imports

On charge Pandas/matplotlib/seaborn pour le coeur du pipeline, et PySpark pour le bonus B3.

In [ ]:
# Installation PySpark (silencieuse) -- pour le bonus en fin de notebook
!pip install pyspark -q

import json
import time
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

print("Imports OK")

## Etape 1/7 -- Charger les 2 sources

On a 2 sources :
- **CSV** : Spotify top tracks (depuis URL GitHub raw)
- **JSON** : profils utilisateurs synthetiques (embedded dans le notebook)

Dans un vrai pipeline, ces sources pourraient etre une API, une BDD, un fichier sur S3...

In [ ]:
# A completer : charger le CSV Spotify
URL_DATASET = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"

try:
    df_tracks = pd.read_csv(URL_DATASET)
    print(f"Spotify charge depuis l'URL : {df_tracks.shape}")
except Exception as e:
    print(f"URL inaccessible ({e}). Fallback synthetique.")
    np.random.seed(42)
    genres = ["pop", "rock", "rap", "latin", "edm", "r&b"]
    artistes = [f"Artist {chr(65 + i)}" for i in range(15)]
    df_tracks = pd.DataFrame({
        "track_id": [f"T{i:05d}" for i in range(5000)],
        "track_name": [f"Song {i}" for i in range(5000)],
        "track_artist": np.random.choice(artistes, 5000),
        "track_popularity": np.random.randint(0, 100, 5000),
        "playlist_genre": np.random.choice(genres, 5000),
        "duration_ms": np.random.randint(120_000, 360_000, 5000),
        "danceability": np.random.uniform(0, 1, 5000),
        "energy": np.random.uniform(0, 1, 5000),
    })
    print(f"Fallback Spotify : {df_tracks.shape}")

df_tracks.head()

In [ ]:
# Generer des profils utilisateurs synthetiques (JSON embedded)
np.random.seed(7)

genres_dispo = ["pop", "rock", "rap", "latin", "edm", "r&b"]
villes = ["Paris", "Nantes", "Lyon", "Marseille", "Toulouse", "Bordeaux"]

profils = []
for i in range(1, 501):
    profils.append({
        "user_id": f"U{i:04d}",
        "age": int(np.random.randint(15, 60)),
        "ville": np.random.choice(villes),
        "genre_prefere": np.random.choice(genres_dispo),
        "abonne_premium": bool(np.random.choice([True, False], p=[0.4, 0.6])),
        "ecoutes_par_jour": int(np.random.poisson(20)),
    })

# Serialiser en JSON puis recharger (simule un vrai fichier JSON)
profils_json_str = json.dumps(profils)
df_users = pd.read_json(StringIO(profils_json_str))

print(f"Profils utilisateurs : {df_users.shape}")
df_users.head()

## Etape 2/7 -- Explorer

Avant de nettoyer, il faut **comprendre** ce qu'on a : shape, types, valeurs manquantes, distributions.

In [ ]:
print("=== TRACKS .info() ===")
df_tracks.info()

print("\n=== TRACKS .describe() ===")
print(df_tracks.describe(include='all').T.head(10))

print("\n=== TRACKS valeurs manquantes ===")
print(df_tracks.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
# A completer : faire la meme exploration sur df_users
print("=== USERS .info() ===")
df_users.info()

print("\n=== USERS .describe() ===")
print(df_users.describe(include='all').T)

print("\n=== USERS valeurs manquantes ===")
print(df_users.isna().sum())

**Questions a se poser** :
- Y a-t-il des colonnes a typer differemment ?
- Combien de valeurs manquantes par colonne dans `tracks` ?
- Y a-t-il des doublons potentiels (memes `track_id` ou meme `track_name + artist`) ?

## Etape 3/7 -- Nettoyer

L'etape la plus longue dans la vraie vie. "Garbage in, garbage out".

In [ ]:
# Nettoyage tracks
df_tracks_clean = df_tracks.copy()

# A completer : supprimer les doublons sur track_id
if "track_id" in df_tracks_clean.columns:
    df_tracks_clean = df_tracks_clean.drop_duplicates(subset=["track_id"])
else:
    df_tracks_clean = df_tracks_clean.drop_duplicates()

# A completer : enlever les lignes ou track_name est manquant
df_tracks_clean = df_tracks_clean.dropna(subset=["track_name"])

# A completer : caster track_popularity en int (au cas ou)
if "track_popularity" in df_tracks_clean.columns:
    df_tracks_clean["track_popularity"] = df_tracks_clean["track_popularity"].astype(int)

print(f"Avant : {len(df_tracks)} lignes")
print(f"Apres : {len(df_tracks_clean)} lignes")
print(f"Supprimees : {len(df_tracks) - len(df_tracks_clean)}")

In [ ]:
# Nettoyage users (deja propres -- mais on verifie)
df_users_clean = df_users.copy().drop_duplicates(subset=["user_id"])

# A completer : s'assurer que age est positif et < 120
df_users_clean = df_users_clean[(df_users_clean["age"] > 0) & (df_users_clean["age"] < 120)]

print(f"Users propres : {len(df_users_clean)} profils")

## Etape 4/7 -- Joindre les 2 sources

On veut analyser **les morceaux ecoutes par les utilisateurs**. Comme on n'a pas de table d'ecoutes reelles, on simule des ecoutes en associant a chaque utilisateur un sous-ensemble aleatoire de morceaux **de son genre prefere**.

C'est notre cle commune : `playlist_genre = genre_prefere`.

In [ ]:
# Generer une table d'ecoutes synthetique pour avoir une vraie jointure
np.random.seed(13)
ecoutes = []
for _, user in df_users_clean.iterrows():
    # chaque user ecoute entre 5 et 20 morceaux
    n = np.random.randint(5, 20)
    candidates = df_tracks_clean[df_tracks_clean["playlist_genre"] == user["genre_prefere"]]
    if len(candidates) == 0:
        candidates = df_tracks_clean
    sample = candidates.sample(n=min(n, len(candidates)), random_state=int(user["age"]))
    for track_id in sample["track_id"].values:
        ecoutes.append({"user_id": user["user_id"], "track_id": track_id})

df_ecoutes = pd.DataFrame(ecoutes)
print(f"Ecoutes generees : {len(df_ecoutes)}")
df_ecoutes.head()

In [ ]:
# A completer : faire 2 merges successifs
# 1) ecoutes <- users  (sur user_id)
# 2) <- tracks  (sur track_id)
df_merged = (
    df_ecoutes
    .merge(df_users_clean, on="user_id", how="left")
    .merge(df_tracks_clean[["track_id", "track_name", "track_artist", "playlist_genre", "track_popularity"]],
           on="track_id", how="left")
)

print(f"Dataset joint : {df_merged.shape}")
df_merged.head()

## Etape 5/7 -- Analyser

On cherche **1 pattern interessant** : par exemple, popularite moyenne des morceaux ecoutes par genre.

In [ ]:
# A completer : groupby + aggregation
# Popularite moyenne des morceaux ecoutes, par genre
popularite_par_genre = (
    df_merged
    .groupby("playlist_genre")["track_popularity"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)

print("=== Popularite moyenne par genre ===")
print(popularite_par_genre.round(1))

In [ ]:
# A completer : un 2e angle d'analyse
# Nombre d'ecoutes par ville (top 10)
ecoutes_par_ville = (
    df_merged.groupby("ville")
    .size()
    .sort_values(ascending=False)
)

print("=== Nombre d'ecoutes par ville ===")
print(ecoutes_par_ville)

## Etape 6/7 -- Visualiser

2 graphs : **histogramme** de la popularite + **scatter** popularite vs age.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Graph 1 : histogramme de la popularite des morceaux ecoutes
axes[0].hist(df_merged["track_popularity"].dropna(), bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Distribution de la popularite des morceaux ecoutes")
axes[0].set_xlabel("track_popularity")
axes[0].set_ylabel("Frequence")
axes[0].axvline(df_merged["track_popularity"].mean(), color="darkred", linestyle="--",
                label=f"moyenne = {df_merged['track_popularity'].mean():.1f}")
axes[0].legend()

# Graph 2 : scatter age vs popularite moyenne ecoutee
# A completer : grouper par age et tracer la moyenne
by_age = df_merged.groupby("age")["track_popularity"].mean()
axes[1].scatter(by_age.index, by_age.values, color="coral", s=40)
axes[1].set_title("Popularite moyenne ecoutee, par age d'utilisateur")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Popularite moyenne")

plt.tight_layout()
plt.show()

In [ ]:
# Graph 3 : bar chart popularite par genre
fig, ax = plt.subplots(figsize=(12, 5))
popularite_par_genre["mean"].plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Popularite moyenne des morceaux ecoutes, par genre")
ax.set_xlabel("Popularite moyenne")
plt.tight_layout()
plt.show()

## Etape 7/7 BONUS B3 -- Refaire en PySpark

On charge les memes donnees dans Spark, on refait l'agregation, on compare les resultats **et on regarde le plan avec `.explain()`**.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, desc

spark = (
    SparkSession.builder
    .appName("B3-J1-Atelier2")
    .getOrCreate()
)

print(f"Spark version : {spark.version}")

In [ ]:
# A completer : recharger df_merged dans Spark
# Indice : spark.createDataFrame(pandas_df)
sdf_merged = spark.createDataFrame(df_merged[["user_id", "track_id", "playlist_genre", "track_popularity", "age", "ville"]])

print(f"Spark DataFrame : {sdf_merged.count()} lignes")
sdf_merged.printSchema()

In [ ]:
# A completer : refaire l'agregation "popularite moyenne par genre" en Spark
popularite_spark = (
    sdf_merged
    .groupBy("playlist_genre")
    .agg(
        avg("track_popularity").alias("popularite_moyenne"),
        count("*").alias("nb_ecoutes"),
    )
    .orderBy(desc("popularite_moyenne"))
)

popularite_spark.show()

In [ ]:
# Comparaison : meme resultat que Pandas ?
print("=== PANDAS ===")
print(popularite_par_genre[["mean", "count"]].round(2))

print("\n=== SPARK ===")
popularite_spark.show(truncate=False)

In [ ]:
# .explain() : qu'est-ce que Spark va executer ?
print("=== PHYSICAL PLAN ===")
popularite_spark.explain()

print("\n=== EXTENDED PLAN ===")
popularite_spark.explain(extended=True)

**Ce qu'on voit dans le plan** :
- `HashAggregate` (1 par genre, partiel puis global) -- c'est notre `groupBy + avg`
- `Exchange hashpartitioning(playlist_genre)` -- le **shuffle** entre les 2 aggregates
- `Sort` final pour le `orderBy`

Sur un cluster, Spark distribuerait l'agregation partielle sur chaque executor, puis ferait le shuffle, puis l'agregation globale. C'est le pattern classique du `combineByKey`.

In [ ]:
# Comparaison perf Pandas vs Spark sur cette agregation
t0 = time.time()
df_merged.groupby("playlist_genre")["track_popularity"].mean()
t_pandas = time.time() - t0

t0 = time.time()
popularite_spark.collect()
t_spark = time.time() - t0

print(f"Pandas : {t_pandas:.4f}s")
print(f"Spark  : {t_spark:.4f}s")
print(f"Ratio Spark/Pandas : {t_spark / max(t_pandas, 0.0001):.1f}x")
print("\nSur ce volume, Pandas gagne. C'est normal -- pas assez de donnees pour amortir l'overhead Spark.")

In [ ]:
spark.stop()
print("Session Spark fermee.")

## Recap + transition projet fil rouge

Vous venez de construire un **mini-pipeline ETL** complet :

```
CSV + JSON  ->  Charger  ->  Explorer  ->  Nettoyer  ->  Joindre  ->  Analyser  ->  Visualiser  ->  [Spark]
(Extract)        (Load)        (Profile)     (Transform)   (Transform)   (Aggregate)   (Communicate)   (Scale)
```

**C'est le squelette de votre projet fil rouge sur 4 jours** :

| Jour | Ce que vous ajoutez |
|---|---|
| J1 (aujourd'hui) | Pipeline ETL + analyse exploratoire |
| J2 | Modele ML baseline (predire une cible) |
| J3 | Dashboard Streamlit pour communiquer les resultats |
| J4 | Deploiement Azure + soutenance |

Le pipeline d'aujourd'hui est la **fondation** : si vos donnees sont sales ou mal jointes, votre ML sera nul et votre dashboard mentira.

## A tester en autonomie

Si vous avez fini en avance ou pour le travail en projet :

1. **Ajouter une 3e source via API** : par exemple, recuperer la meteo d'une ville avec `requests.get("https://api.open-meteo.com/...")`, et croiser les ecoutes avec la meteo.
2. **Filtre temporel** : ajouter un champ `date_ecoute` synthetique a `df_ecoutes`, puis analyser la repartition des ecoutes par heure / jour de semaine.
3. **Comparer abonnes premium vs gratuits** : popularite moyenne, diversite de genres ecoutes, nombre d'ecoutes par jour. Avec un boxplot seaborn.